# EcoSort AI

Run all cells to inspect the saved results and open the English upload/webcam demo. Training and dataset rebuilding are optional and disabled by default. The model is ImageNet-initialized MobileNetV3Large, trained on project images.


## 1. Project structure

Six modules: prepare_data.ipynb collects and cleans images; train.ipynb trains; evaluate.ipynb evaluates; classifier.ipynb predicts; disposal_rules.ipynb supplies local guidance; app.ipynb builds the interface. data/ contains images and manifests. model/ contains model.keras, settings.json and results.json.


In [ ]:
from pathlib import Path
import json, sys, types
PROJECT = Path.cwd()
assert (PROJECT / 'app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'

def load_notebook(name):
    if name in sys.modules:
        return sys.modules[name]
    path = PROJECT / (name + '.ipynb')
    notebook = json.loads(path.read_text())
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for dependency in notebook['metadata'].get('dependencies', []):
            load_notebook(dependency)
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code' and 'module' in cell.get('metadata', {}).get('tags', []):
                exec(compile(''.join(cell['source']), str(path), 'exec'), module.__dict__)
    except Exception:
        sys.modules.pop(name, None)
        raise
    return module

from pathlib import Path
import json
import subprocess
import sys

assert Path('app.ipynb').is_file(), 'Open this notebook from EcoSort-Demo.'
results = json.loads(Path('model/results.json').read_text())
print('Training split:', results['training']['split_counts'])
print('Saved evaluation:', results['evaluation']['metrics'])


## 2. Collect, map, clean and split data

prepare_data.ipynb restores the original expanded-data preparation logic. It combines the fixed original 20,000-image selection with 147,285 Garbage265 metadata entries, maps labels, decodes and hashes images, groups duplicates, excludes conflicting labels and splits groups 80/10/10. Original source labels are not Sydney disposal rules.

The original 20,000-image selection is retained as an explicit input in data/source_lists/original_20000.csv, not selected again from all upstream files. Add --download only to restore missing sources. Rebuilding writes to a NEW folder and never silently replaces the deployed split or model.


In [ ]:
REBUILD_DATA = False
if REBUILD_DATA:
    saved_arguments = sys.argv[:]
    try:
        sys.argv = ['prepare_data', '--download', '--output', 'data/prepared']
        load_notebook('prepare_data').main()
    finally:
        sys.argv = saved_arguments
else:
    report = json.loads(Path('data/dataset_verification.json').read_text())
    print('Candidates:', report['raw_candidate_images'], 'Retained:', report['retained_images'])


## 3. Train on the project dataset

The fixed data/split_manifest.csv supplies every training image. Train the new head over frozen ImageNet features, then fine-tune the last 60 backbone layers except batch normalization. Class weights and training-only augmentation reduce imbalance and overfitting. Validation macro-F1 selects the model. A new training run writes to model_new/ and refuses to overwrite an existing run; it does not replace the deployed model automatically.


In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    saved_arguments = sys.argv[:]
    try:
        sys.argv = ['train', '--output', 'model_new']
        load_notebook('train').main()
    finally:
        sys.argv = saved_arguments
else:
    print('Completed training:', results['training']['selected'])


## 4. Evaluate the deployed model

The saved validation/test results are measured dataset results, not a guarantee for new real-world photos. evaluate.ipynb evaluates all fixed validation/test images, checks inputs and rules, and updates model/results.json. It does not train on test images.


In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    load_notebook('evaluate').main()
results = json.loads(Path('model/results.json').read_text())
print(json.dumps(results['evaluation']['metrics'], indent=2))


## 5. Material classification and local guidance

The neural network predicts 17 broad categories. It cannot verify bottle/bag identity, emptiness or hidden hazards. Confirm the exact item in the interface. This confirmation changes disposal guidance, not the model weights.


In [ ]:
disposal_guidance = load_notebook('disposal_rules').disposal_guidance

for item in ('Not confirmed', 'Empty plastic water/drink bottle', 'Soft plastic bag or wrapper'):
    print(item, '->', disposal_guidance('plastic', item)[0])


## 6. Open the demo

Upload or capture a photo, optionally confirm its item type and food-scraps service access, and select Identify item. Low scores, small top-two margins and designated classes require review. The server runs only on this computer.


In [ ]:
launch = load_notebook('app').launch

demo = launch(prevent_thread_lock=True)
